In [ ]:
import os
from bs4 import BeautifulSoup
import requests
import pandas as pd
from pathlib import Path
import time
from tqdm.notebook import tqdm
import re

In [ ]:
def download_image(image_url, filename):
    path = Path("images") / filename
    os.makedirs(path.parent, exist_ok=True)
    if path.exists():
        return
    try:
        response = requests.get(image_url, stream=True)
        response.raise_for_status()

        with open(path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

    except Exception as e:
        print(f"Ошибка при скачивании {image_url}: {e}")

In [ ]:
def clean_profile_data(profile_data):
    for key in ["Обо мне", "Кого ищу"]:
        if key in profile_data and pd.notna(profile_data[key]):
            value = re.sub(r"[\r\n\t]+", " ", profile_data[key])
            value = re.sub(r"\s+", " ", value)
            profile_data[key] = value.strip()

    for key in ["Рост, см", "Вес, кг"]:
        if key in profile_data and profile_data[key] == "-":
            profile_data[key] = pd.NA

    return profile_data

In [ ]:
def parse_profile_page(profile_url, profile_id):
    try:
        response = requests.get(profile_url)
        soup = BeautifulSoup(response.text, "html.parser")

        gender_elem = soup.find("meta", property="profile:gender")
        if not gender_elem:
            return None

        name, age = soup.find("span", class_="usrinf-name").text.split(", ")

        profile_data = {
            "id": profile_id,
            "Имя": name.strip(),
            "Возраст": age.strip(),
            "Пол": gender_elem.get("content"),
        }

        for title, value in zip(
            soup.select("p.usrinf-title"), soup.select("p.usrinf-txt")
        ):
            profile_data[title.text.strip()] = value.text.strip()

        # photo_elements = soup.select('div.photo-content > div > a > meta[itemprop=contentUrl]')
        # for i, photo_elem in enumerate(photo_elements):
        #     photo_url = photo_elem.get('content')
        #     download_image(photo_url, f"{profile_id}/{i}.jpg")

        clean_profile_data(profile_data)

        return profile_data

    except Exception as e:
        print(f"Ошибка парсинга профиля {profile_id}: {e}")
        return None

In [ ]:
def scrape_beboo_profiles(existing_ids, max_pages=10):
    base_url = "https://m.beboo.ru/"
    profiles = []

    for page in tqdm(range(1, max_pages + 1), desc="Страницы поиска"):
        try:
            search_params = {
                "status": "all",
                "lookFor": 0,
                "f": 0,
                "countryId": 104,
                "regionId": -1,
                "townId": -1,
                "startAge": 18,
                "endAge": 80,
                "reason": 0,
                "page": page,
            }

            response = requests.get(base_url + "search", params=search_params)
            soup = BeautifulSoup(response.text, "html.parser")

            skipped = 0
            start_time = time.time()

            for profile_elem in tqdm(
                soup.select("a.srh-info"), desc=f"Профили стр. {page}", leave=False
            ):
                profile_id = int(profile_elem.get("href").split("/")[-1].split("?")[0])
                if profile_id in existing_ids:
                    skipped += 1
                    continue

                profile_data = parse_profile_page(
                    base_url + profile_elem.get("href"), profile_id
                )
                if profile_data is None:
                    skipped += 1
                else:
                    profiles.append(profile_data)
                    existing_ids.add(profile_id)

                time.sleep(0.1)

            print(
                f"Стр. {page}: пропущено {skipped}, прошло {time.time() - start_time:.2f}с"
            )

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")
            continue

    return profiles

In [ ]:
existing_ids = set(pd.read_csv("data/data.tsv", sep="\t", usecols=["id"])["id"])
new_profiles = scrape_beboo_profiles(existing_ids, max_pages=100)

columns = [
    "id",
    "Имя",
    "Возраст",
    "Пол",
    "Город",
    "Знакомлюсь для",
    "Семейное положение",
    "Дети",
    "Курение",
    "Алкоголь",
    "Доход",
    "Проживание",
    "Наличие автомобиля",
    "Знание языков",
    "Рост, см",
    "Тело",
    "Вес, кг",
    "Цвет волос",
    "Цвет глаз",
    "Обо мне",
    "Кого ищу",
]
new_df = pd.DataFrame(new_profiles, columns=columns)
new_df.to_csv("data/new.tsv", sep="\t", index=False, encoding="utf-8")